# 01 - Introducción: cargar y explorar el histórico

Este notebook asume que ya existe `Data/impo_historico.parquet` (correr
`python Data/descargar_historico_impo.py --actualizar` desde la raíz del repo
si todavía no existe). Cubre lo mínimo para empezar:

- Qué es cada columna.
- Cómo se representan las fechas (`PERIODO`/`FECHA`, texto `YYYYMM`) y cómo
  convertirlas a fechas reales.
- Consultas básicas: contar filas, buscar un importador, sumar por año.

Motor usado acá: **DuckDB** sobre el Parquet directo (no hace falta cargar
nada a RAM de antemano; ver el notebook 03 para por qué esto importa en una
máquina con poca memoria).

In [ ]:
import duckdb

# Ruta relativa a este notebook (notebooks/../Data/...)
HISTORICO = "../Data/impo_historico.parquet"

con = duckdb.connect()
# Limite explicito de RAM: este dataset puede tener cientos de millones de
# filas, y sin esto DuckDB intenta usar la RAM libre que encuentre (ver
# notebook 03 para el detalle de por que esto importa en una maquina chica).
# temp_directory es igual de necesario: sin un lugar donde derramar, al tocar
# el limite DuckDB no tiene alternativa y la consulta falla en vez de
# resolverse (mas lento) usando disco.
con.execute("PRAGMA memory_limit='1.5GB'")
con.execute("PRAGMA temp_directory='../Data'")
con.execute(f"CREATE VIEW impo AS SELECT * FROM read_parquet('{HISTORICO}')")


## Columnas disponibles

In [ ]:
con.execute("DESCRIBE impo").fetchdf()


## Cuánto abarca el histórico

`PERIODO` es el mes de origen de cada fila, como texto `YYYYMM` (por ejemplo
`"202508"` para agosto de 2025). No es un tipo fecha: es texto, tal cual viene
del archivo `.lst` de ARCA.

In [ ]:
con.execute('''
    SELECT count(*) AS filas,
           count(DISTINCT PERIODO) AS meses,
           min(PERIODO) AS desde,
           max(PERIODO) AS hasta
    FROM impo
''').fetchdf()


## Fechas: de texto `YYYYMM` a `DATE`

`strptime(PERIODO || '01', '%Y%m%d')` arma un `YYYYMMDD` pegando el día 01 y
lo parsea como fecha. Sirve para filtrar por rango, agrupar por año/mes con
funciones de fecha, graficar series de tiempo, etc.

In [ ]:
con.execute('''
    SELECT PERIODO,
           strptime(PERIODO || '01', '%Y%m%d')::DATE AS fecha,
           count(*) AS filas
    FROM impo
    GROUP BY PERIODO
    ORDER BY PERIODO
    LIMIT 5
''').fetchdf()


## Filtrar por rango de fechas

Comparar `PERIODO` como texto `YYYYMM` alcanza para rangos de mes completo (no hace falta convertir a `DATE` para esto):

In [ ]:
con.execute('''
    SELECT count(*) AS filas, sum(FOB_TOTAL_USD) AS fob_total_usd
    FROM impo
    WHERE PERIODO BETWEEN '202301' AND '202312'
''').fetchdf()


## Total FOB por año

In [ ]:
con.execute('''
    SELECT substr(PERIODO, 1, 4) AS anio,
           count(*) AS filas,
           round(sum(FOB_TOTAL_USD)) AS fob_total_usd
    FROM impo
    GROUP BY 1
    ORDER BY 1
''').fetchdf()


## Buscar un importador

`FOB_UNITARIO_USD` es el FOB del ítem (se repite en cada línea de tributo);
`FOB_TOTAL_USD` es el FOB de toda la declaración (se repite en todos sus
ítems). Para no duplicar el total al sumar, agrupar primero por
declaración+ítem con `any_value` (ver notebook 02).

In [ ]:
con.execute('''
    SELECT PERIODO, DESTINACION, NUM_ITEM, NOMBRE_IMPORTADOR, POS_NCM, FOB_UNITARIO_USD
    FROM impo
    WHERE upper(NOMBRE_IMPORTADOR) LIKE '%ROCHE%'
    ORDER BY PERIODO
    LIMIT 20
''').fetchdf()


## Siguiente paso

`02_analisis_intermedio.ipynb`: fechas más avanzadas (series de tiempo,
crecimiento interanual), decodificar países/aduanas/unidades con
`codigos/codigos_arca.py`, y rankings por NCM.